# Table 1 matching
Minimal preprocessing for donor ranking files.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, energy_distance

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUT_DIR = DATA_DIR / "processed_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2017
TOP_K = 65
MAX_SAMPLES = 800
SINKHORN_EPS = 0.05
SINKHORN_ITERS = 200

CANDIDATE_FEATURES = [
    "wind_speed",
    "wind_direction",
    "temperature",
    "turbulence_intensity",
    "std_wind_direction",
]

FEATURE_WEIGHTS = {
    "wind_speed": 80.0,
    "wind_direction": 5.0,
    "temperature": 10.0,
    "turbulence_intensity": 3.0,
    "std_wind_direction": 2.0,
}

In [2]:
def find_turbine_files(data_dir: Path, year: int):
    rx = re.compile(rf"^Turbine(\d+)_{year}\.csv$", re.IGNORECASE)
    files = {}
    for p in data_dir.iterdir():
        m = rx.match(p.name)
        if m:
            files[int(m.group(1))] = p
    return dict(sorted(files.items()))

def clean_array(x):
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy(dtype=float)
    return x[np.isfinite(x)]

def subsample_sorted(x, max_samples=MAX_SAMPLES):
    x = np.sort(clean_array(x))
    if len(x) <= max_samples:
        return x
    idx = np.linspace(0, len(x) - 1, max_samples).round().astype(int)
    return x[idx]

def sinkhorn_distance_1d(x, y, eps=SINKHORN_EPS, n_iter=SINKHORN_ITERS):
    x = subsample_sorted(x)
    y = subsample_sorted(y)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    a = np.full(len(x), 1.0 / len(x))
    b = np.full(len(y), 1.0 / len(y))
    C = np.abs(x[:, None] - y[None, :])
    K = np.exp(-C / max(eps, 1e-8))
    u = np.ones(len(x))
    v = np.ones(len(y))
    for _ in range(n_iter):
        Kv = K @ v
        Kv[Kv == 0] = 1e-300
        u = a / Kv
        KTu = K.T @ u
        KTu[KTu == 0] = 1e-300
        v = b / KTu
    P = (u[:, None] * K) * v[None, :]
    return float(np.sum(P * C))

def average_ks(df_a, df_b, features):
    vals = []
    for col in features:
        a = clean_array(df_a[col])
        b = clean_array(df_b[col])
        if len(a) == 0 or len(b) == 0:
            continue
        vals.append(ks_2samp(a, b).statistic)
    return np.nan if len(vals) == 0 else float(np.mean(vals))

def weighted_ks(df_a, df_b, features, weights):
    vals, ws = [], []
    for col in features:
        a = clean_array(df_a[col])
        b = clean_array(df_b[col])
        if len(a) == 0 or len(b) == 0:
            continue
        vals.append(ks_2samp(a, b).statistic)
        ws.append(weights[col])
    if len(vals) == 0:
        return np.nan
    ws = np.array(ws, dtype=float)
    ws = ws / ws.sum()
    return float(np.dot(ws, vals))

def average_energy(df_a, df_b, features):
    vals = []
    for col in features:
        a = clean_array(df_a[col])
        b = clean_array(df_b[col])
        if len(a) == 0 or len(b) == 0:
            continue
        vals.append(energy_distance(a, b))
    return np.nan if len(vals) == 0 else float(np.mean(vals))

def average_sinkhorn(df_a, df_b, features):
    vals = []
    for col in features:
        a = clean_array(df_a[col])
        b = clean_array(df_b[col])
        if len(a) == 0 or len(b) == 0:
            continue
        vals.append(sinkhorn_distance_1d(a, b))
    return np.nan if len(vals) == 0 else float(np.mean(vals))

In [3]:
files = find_turbine_files(DATA_DIR, YEAR)
if not files:
    raise FileNotFoundError(f"No turbine files found in {DATA_DIR} for year {YEAR}.")

sample_df = pd.read_csv(next(iter(files.values())))
features = [c for c in CANDIDATE_FEATURES if c in sample_df.columns]
if not features:
    raise ValueError("None of the candidate features were found in the turbine files.")

weights = {k: FEATURE_WEIGHTS[k] for k in features}
turbines = {tid: pd.read_csv(path) for tid, path in files.items()}

print("Loaded turbines:", len(turbines))
print("Features:", features)

Loaded turbines: 66
Features: ['wind_speed', 'wind_direction', 'temperature', 'turbulence_intensity', 'std_wind_direction']


In [4]:
rows = []
ids = sorted(turbines.keys())

for target in ids:
    df_t = turbines[target]
    for donor in ids:
        if donor == target:
            continue
        df_d = turbines[donor]
        rows.append({
            "target": target,
            "donor": donor,
            "weighted_ks": weighted_ks(df_t, df_d, features, weights),
            "mean_ks": average_ks(df_t, df_d, features),
            "marginal_energy": average_energy(df_t, df_d, features),
            "sinkhorn_wasserstein": average_sinkhorn(df_t, df_d, features),
        })

pairwise = pd.DataFrame(rows).sort_values(["target", "donor"]).reset_index(drop=True)
pairwise.to_csv(OUT_DIR / f"pairwise_matching_{YEAR}.csv", index=False)
pairwise.head()

C:\Users\achokhachian3\AppData\Local\Temp\ipykernel_61232\2377400900.py:35: RuntimeWarning: overflow encountered in divide
  u = a / Kv
C:\Users\achokhachian3\AppData\Local\Temp\ipykernel_61232\2377400900.py:36: RuntimeWarning: invalid value encountered in matmul
  KTu = K.T @ u


C:\Users\achokhachian3\AppData\Local\Temp\ipykernel_61232\2377400900.py:38: RuntimeWarning: overflow encountered in divide
  v = b / KTu
C:\Users\achokhachian3\AppData\Local\Temp\ipykernel_61232\2377400900.py:33: RuntimeWarning: invalid value encountered in matmul
  Kv = K @ v


C:\Users\achokhachian3\AppData\Local\Temp\ipykernel_61232\2377400900.py:39: RuntimeWarning: invalid value encountered in multiply
  P = (u[:, None] * K) * v[None, :]


,target,donor,weighted_ks,mean_ks,marginal_energy,sinkhorn_wasserstein
0,1,2,0.061651,0.116697,0.864982,0.291091
1,1,3,0.106043,0.088509,0.483168,0.228032
2,1,4,0.123918,0.158754,1.650844,0.233935
3,1,5,0.039841,0.097617,0.996436,NaN
4,1,6,0.098502,0.147511,1.807804,0.395894


In [5]:
loc_path = DATA_DIR / "location.csv"
if not loc_path.exists():
    raise FileNotFoundError(f"Missing {loc_path}")

loc = pd.read_csv(loc_path, encoding="utf-8-sig")
loc.columns = [str(c).strip().replace("\ufeff", "") for c in loc.columns]

id_col = next((c for c in loc.columns if c.lower() in ["wt", "turbine", "id"]), None)
lon_col = next((c for c in loc.columns if c.lower() == "longitude"), None)
lat_col = next((c for c in loc.columns if c.lower() == "latitude"), None)

print("Location columns:", list(loc.columns))
print(loc.head())

if id_col is None or lon_col is None or lat_col is None:
    raise ValueError(
        f"location.csv columns were read as {list(loc.columns)}; expected WT (or turbine/id), Longitude, Latitude"
    )

loc = loc[[id_col, lon_col, lat_col]].copy()
loc.columns = ["target", "Longitude", "Latitude"]
loc["target"] = pd.to_numeric(loc["target"], errors="coerce")
loc["Longitude"] = pd.to_numeric(loc["Longitude"], errors="coerce")
loc["Latitude"] = pd.to_numeric(loc["Latitude"], errors="coerce")
loc = loc.dropna().copy()
loc["target"] = loc["target"].astype(int)

geo_rows = []
for _, row_i in loc.iterrows():
    for _, row_j in loc.iterrows():
        if row_i["target"] == row_j["target"]:
            continue
        d = np.sqrt(
            (row_i["Longitude"] - row_j["Longitude"]) ** 2 +
            (row_i["Latitude"] - row_j["Latitude"]) ** 2
        )
        geo_rows.append({
            "target": int(row_i["target"]),
            "donor": int(row_j["target"]),
            "geo_distance": float(d),
        })

geo_pairwise = (
    pd.DataFrame(geo_rows)
    .sort_values(["target", "geo_distance", "donor"])
    .reset_index(drop=True)
)

geo_pairwise.to_csv(OUT_DIR / "pairwise_geographic_distance.csv", index=False)
geo_pairwise.head()

Location columns: ['WT', 'Longitude', 'Latitude']
   WT  Longitude  Latitude
0   1   0.549103  0.100883
1   2   0.545652  0.100072
2   3   0.542383  0.102782
3   4   0.539092  0.103423
4   5   0.536123  0.105425


,target,donor,geo_distance
0,1,2,0.003546
1,1,52,0.005286
2,1,3,0.006983
3,1,4,0.010329
4,1,53,0.010602


In [6]:
def ranking_table(df, score_col, top_k=TOP_K):
    out = []
    for target, g in df.groupby("target"):
        g = g.sort_values([score_col, "donor"]).head(top_k).reset_index(drop=True)
        row = {"target": target}
        for r, (_, rr) in enumerate(g.iterrows(), start=1):
            row[f"donor_{r}"] = int(rr["donor"])
            row[f"score_{r}"] = float(rr[score_col])
        out.append(row)
    return pd.DataFrame(out).sort_values("target").reset_index(drop=True)

metric_map = {
    "weighted_ks": "matching_weighted_ks.csv",
    "mean_ks": "matching_mean_ks.csv",
    "marginal_energy": "matching_marginal_energy.csv",
    "sinkhorn_wasserstein": "matching_sinkhorn_wasserstein.csv",
}

for metric, fname in metric_map.items():
    ranking_table(pairwise[["target", "donor", metric]].copy(), metric).to_csv(OUT_DIR / fname, index=False)

ranking_table(geo_pairwise[["target", "donor", "geo_distance"]].copy(), "geo_distance").to_csv(
    OUT_DIR / "matching_geographic_distance.csv", index=False
)

summary = pd.DataFrame({
    "file": [
        f"pairwise_matching_{YEAR}.csv",
        "pairwise_geographic_distance.csv",
        "matching_weighted_ks.csv",
        "matching_mean_ks.csv",
        "matching_marginal_energy.csv",
        "matching_sinkhorn_wasserstein.csv",
        "matching_geographic_distance.csv",
    ]
})
summary.to_csv(OUT_DIR / "matching_files_created.csv", index=False)
summary

,file
0,pairwise_matching_2017.csv
1,pairwise_geographic_distance.csv
2,matching_weighted_ks.csv
3,matching_mean_ks.csv
4,matching_marginal_energy.csv
5,matching_sinkhorn_wasserstein.csv
6,matching_geographic_distance.csv
